# Model Comparison with Permutation Tests and CDS

This notebook demonstrates how to:
1. Compare ECL across multiple models using the paired permutation test (Algorithm 5)
2. Decompose influence profiles using Context Decay Spectroscopy (CDS)

We use three synthetic models with different decay lengths to simulate short-, medium-, and long-context architectures.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from ecl.models.base import SyntheticModel
from ecl.influence import compute_influence_profile
from ecl.ecl import ECL, cumulative_influence
from ecl.estimation import permutation_test
from ecl.cds import fit_cds, select_n_components
from ecl.perturbations import RandomSubstitution

In [ ]:
# Three models with different context utilization
models = {
    "CNN (short)": SyntheticModel(500, 32, decay_length=20),
    "Transformer (medium)": SyntheticModel(500, 32, decay_length=80),
    "SSM (long)": SyntheticModel(500, 32, decay_length=150),
}
rng = np.random.default_rng(123)
ref, n_loci, n_seqs = 250, 30, 10

In [ ]:
# Collect per-locus ECL for each model
pert = RandomSubstitution()
ecl_per_model = {}

for name, model in models.items():
    ecl_loci = np.zeros(n_loci)
    for j in range(n_loci):
        seqs = rng.integers(0, 4, size=(n_seqs, 500)).astype(np.int8)
        d, infl = compute_influence_profile(
            model, seqs, ref, max_distance=200, positions_per_distance=2,
            perturbation=pert, rng=rng, show_progress=False,
        )
        ecl_loci[j] = ECL(d, infl, beta=0.9)
    ecl_per_model[name] = ecl_loci
    print(f"{name}: mean ECL_0.9 = {ecl_loci.mean():.0f} +/- {ecl_loci.std():.0f} bp")

In [ ]:
# Permutation test: Transformer vs CNN
ecl_a = ecl_per_model["CNN (short)"]
ecl_b = ecl_per_model["Transformer (medium)"]

mean_diff, p_val, ci = permutation_test(ecl_b, ecl_a, n_permutations=2000, rng=rng)
print(f"Delta ECL (Transformer - CNN) = {mean_diff:.1f} bp, p = {p_val:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ecl_b - ecl_a, bins=15, alpha=0.7, edgecolor='black')
ax.axvline(mean_diff, color='red', linewidth=2, label=f'Mean={mean_diff:.0f}')
ax.axvline(0, color='black', ls='--')
ax.set_xlabel('Delta ECL_0.9 (bp)')
ax.set_title(f'Paired Differences (p={p_val:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CDS analysis
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e41a1c', '#377eb8', '#4daf4a']

for (name, model), color in zip(models.items(), colors):
    seqs = rng.integers(0, 4, size=(20, 500)).astype(np.int8)
    d, infl = compute_influence_profile(
        model, seqs, ref, max_distance=200, positions_per_distance=3,
        perturbation=pert, rng=rng, show_progress=False,
    )
    best_K, results = select_n_components(d, infl, max_K=3)
    cds = results[best_K - 1]
    ax.semilogy(d, infl, color=color, alpha=0.3)
    ax.semilogy(d, cds['fitted'], color=color, linewidth=2, label=f'{name} (K={best_K})')
    print(f"{name}: K={best_K}, decay rates={cds['decay_rates']}")

ax.set_xlabel('Distance (bp)')
ax.set_ylabel('Influence (log)')
ax.set_title('Context Decay Spectroscopy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()